# TLR4 Molecular Docking Pipeline

## Overview

This notebook implements a structure-based molecular docking pipeline for the human
TLR4–MD2 complex. Starting from the crystal structure (PDB ID: 3FXI), the receptor
is prepared, a ligand is built from SMILES, and AutoDock Vina is used to predict
binding poses and affinity scores in the MD-2 hydrophobic pocket.

The docking pipeline in this notebook is used in Notebook 03 to structurally validate
ML-predicted TLR4 ligands.

## Workflow
1. Download and parse TLR4–MD2 structure (PDB ID: 3FXI)
2. Extract and clean receptor chains (Chain A: TLR4, Chain C: MD-2)
3. Calculate MD-2 binding pocket center for docking grid definition
4. Prepare ligand from SMILES — 3D structure generation and format conversion
5. Run molecular docking with AutoDock Vina
6. Extract binding affinity scores and visualize docked complex

## Tools
- **BioPython** — structure parsing and processing
- **RDKit** — ligand preparation and cheminformatics
- **Open Babel** — file format conversion to PDBQT
- **AutoDock Vina** — molecular docking
- **Py3Dmol** — 3D visualization of protein–ligand complex

## Target
- **Receptor:** Human TLR4–MD2 complex (PDB ID: 3FXI)
- **Binding site:** MD-2 hydrophobic pocket (Chain C)
- **Source:** Protein Data Bank (https://www.rcsb.org)

## Step 1: Install Dependencies

- BioPython: protein structure parsing and manipulation


In [2]:
!pip install biopython -q

print("Dependencies installed successfully.")


Dependencies installed successfully.



## Step 2: Upload TLR4–MD2 Receptor Structure

Upload the structure file downloaded from the Protein Data Bank (PDB ID: 3FXI).

In [3]:
from google.colab import files
uploaded = files.upload()


Saving 3FXI.cif to 3FXI.cif


## Step 3: Load Structure

In [4]:
from Bio.PDB import MMCIFParser

parser = MMCIFParser(QUIET=True)
structure = parser.get_structure("3FXI", "3FXI.cif")

print("Structure loaded successfully.")

Structure loaded successfully.


## Step 4: Inspect Receptor Chains

Lists all chain IDs and residue counts to confirm which chains correspond
to TLR4 (Chain A) and MD-2 (Chain C).


In [5]:
for model in structure:
    for chain in model:
        print("Chain ID:", chain.id)

for model in structure:
    for chain in model:
        residues = [res for res in chain if res.id[0] == " "]
        print(f"Chain {chain.id} has {len(residues)} amino acid residues")

Chain ID: A
Chain ID: B
Chain ID: C
Chain ID: D
Chain ID: E
Chain ID: F
Chain ID: G
Chain ID: H
Chain ID: I
Chain ID: J
Chain A has 601 amino acid residues
Chain B has 601 amino acid residues
Chain C has 140 amino acid residues
Chain D has 140 amino acid residues
Chain E has 0 amino acid residues
Chain F has 0 amino acid residues
Chain G has 0 amino acid residues
Chain H has 0 amino acid residues
Chain I has 0 amino acid residues
Chain J has 0 amino acid residues



## Step 5: Extract and Clean TLR4–MD2 Receptor

- Keeps only Chain A (TLR4) and Chain C (MD-2)
- Removes heteroatoms, ligands, and non-standard residues
- Output: TLR4_MD2_clean_AC.pdb

In [6]:
from Bio.PDB import PDBIO, Select

class ReceptorSelect(Select):

    def accept_chain(self, chain):
        return chain.id in ["A", "C"]

    def accept_residue(self, residue):
        return residue.id[0] == " "

io = PDBIO()
io.set_structure(structure)
io.save("TLR4_MD2_clean_AC.pdb", ReceptorSelect())

print("Final cleaned receptor saved as TLR4_MD2_clean_AC.pdb")

Final cleaned receptor saved as TLR4_MD2_clean_AC.pdb


## Step 6: Verify Cleaned Receptor

Confirms only Chain A and Chain C are present and checks residue counts.
Input: TLR4_MD2_clean_AC.pdb


In [7]:
from Bio.PDB import PDBParser

parser = PDBParser(QUIET=True)
final_structure = parser.get_structure("final", "TLR4_MD2_clean_AC.pdb")

for model in final_structure:
    for chain in model:
        residues = [res for res in chain]
        print(f"Chain {chain.id} has {len(residues)} residues")

Chain A has 601 residues
Chain C has 140 residues


## Step 7: Calculate MD-2 Binding Pocket Center

Extracts atomic coordinates from MD-2 (Chain C) and computes the geometric center.
This defines the docking grid center for AutoDock Vina, ensuring docking targets
the biologically relevant binding site.

In [8]:
import numpy as np
from Bio.PDB import PDBParser

parser = PDBParser(QUIET=True)
structure = parser.get_structure("final", "TLR4_MD2_clean_AC.pdb")

md2_atoms = []

for model in structure:
    for chain in model:
        if chain.id == "C":
            for residue in chain:
                for atom in residue:
                    md2_atoms.append(atom.coord)

md2_atoms = np.array(md2_atoms)

center = md2_atoms.mean(axis=0)

print("Docking grid center (x, y, z):")
print(center)

Docking grid center (x, y, z):
[29.385881   1.2959616 20.486185 ]


## Step 8: Define Docking Grid Parameters

Defines the search space centered on the MD-2 pocket:
- Center: x=29.39, y=1.30, z=20.49
- Box size: 24 x 24 x 24 Angstrom (reduced from 28 to focus on MD-2 pocket)

In [9]:
center_x = 29.39
center_y = 1.30
center_z = 20.49

size_x = 24
size_y = 24
size_z = 24

print(f"Grid center: ({center_x}, {center_y}, {center_z}), Box: {size_x}x{size_y}x{size_z} Å")


Grid center: (29.39, 1.3, 20.49), Box: 24x24x24 Å


## Step 9: Install Docking Tools

- AutoDock Vina: molecular docking
- Open Babel: file format conversion
- RDKit: ligand preparation


In [10]:
!pip install vina
!apt-get install -y autodock-vina
!apt-get install -y openbabel

!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 41.1 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libboost-filesystem1.74.0 libboost-program-options1.74.0
  libboost-thread1.74.0
Suggested packages:
  autodock autogrid
The following NEW packages will be installed:
  autodock-vina libboost-filesystem1.74.0 libboost-program-options1.74.0
  libboost-thread1.74.0
0 upgraded, 4 newly installed, 0 to remove and 2 not upgraded.
Need to get 1,120 kB of archives.
After this operation, 7,537 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libboost-filesystem1.74.0 amd64 1.74.0-14ubuntu3 [264 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libboost-program-options1.74.0 amd64 1.74.0-14ubuntu3 [311 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libboost-thread1.74.0 amd64 1.74.0-14ubuntu3 [262 k

## Step 10: Convert Receptor to PDBQT

Converts TLR4_MD2_clean_AC.pdb to receptor.pdbqt using Open Babel.
Adds AutoDock atom types and assigns partial charges.


In [11]:
!obabel TLR4_MD2_clean_AC.pdb -O receptor.pdbqt -xr 2>/dev/null
print("Receptor converted to PDBQT successfully.")

Receptor converted to PDBQT successfully.


In [12]:
!head receptor.pdbqt

REMARK  Name = TLR4_MD2_clean_AC.pdb
REMARK                            x       y       z     vdW  Elec       q    Type
REMARK                         _______ _______ _______ _____ _____    ______ ____
ATOM      1  N   GLU A  27      20.646  27.674  43.260  0.00  0.00    +0.000 NA
ATOM      2  CA  GLU A  27      19.751  26.838  42.401  0.00  0.00    +0.000 C 
ATOM      3  C   GLU A  27      20.496  26.310  41.177  0.00  0.00    +0.000 C 
ATOM      4  O   GLU A  27      20.032  26.490  40.051  0.00  0.00    +0.000 OA
ATOM      5  CB  GLU A  27      18.549  27.657  41.908  0.00  0.00    +0.000 C 
ATOM      6  CG  GLU A  27      17.771  28.389  42.982  0.00  0.00    +0.000 C 
ATOM      7  CD  GLU A  27      16.735  29.310  42.387  0.00  0.00    +0.000 C 


## Step 11: Prepare Ligand from SMILES

Converts SMILES to a 3D structure using Open Babel (--gen3d).
Output: ligand.pdb

In [13]:
# Save SMILES to file
with open("ligand.smi", "w") as f:
    f.write("CCCCCCCCCCCCCC(=O)N[C@@H](COCC[C@H](CCCCCCC)OC(=O)CCCCCCCCCCC)COP(=O)(O)OCCNC(=O)C(Cc1ccc(O)cc1)C(=O)O")

# Generate 3D structure and minimize energy
!obabel ligand.smi -O ligand.pdb --gen3d --minimize

1 molecule converted


In [14]:
!head ligand.pdb

COMPND    UNNAMED
AUTHOR    GENERATED BY OPEN BABEL 3.1.1
HETATM    1  C   UNL     1      36.018  -4.711   0.539  1.00  0.00           C  
HETATM    2  C   UNL     1      35.440  -3.435   1.125  1.00  0.00           C  
HETATM    3  C   UNL     1      33.915  -3.480   1.136  1.00  0.00           C  
HETATM    4  C   UNL     1      33.333  -2.200   1.724  1.00  0.00           C  
HETATM    5  C   UNL     1      31.810  -2.257   1.729  1.00  0.00           C  
HETATM    6  C   UNL     1      31.225  -0.979   2.315  1.00  0.00           C  
HETATM    7  C   UNL     1      29.704  -1.046   2.316  1.00  0.00           C  
HETATM    8  C   UNL     1      29.113   0.227   2.902  1.00  0.00           C  


## Step 12: Convert Ligand to PDBQT

Converts ligand.pdb to ligand.pdbqt, assigning AutoDock atom types
and defining rotatable bonds for flexible docking.

In [15]:
!obabel ligand.pdb -O ligand.pdbqt

1 molecule converted


In [16]:
!head ligand.pdbqt

REMARK  Name = ligand.pdb
REMARK  53 active torsions:
REMARK  status: ('A' for Active; 'I' for Inactive)
REMARK    1  A    between atoms: C_1  and  C_2
REMARK    2  A    between atoms: C_2  and  C_3
REMARK    3  A    between atoms: C_3  and  C_4
REMARK    4  A    between atoms: C_4  and  C_5
REMARK    5  A    between atoms: C_5  and  C_6
REMARK    6  A    between atoms: C_6  and  C_7
REMARK    7  A    between atoms: C_7  and  C_8


## Step 13: Molecular Docking with AutoDock Vina

Docks the ligand into the TLR4–MD2 binding site.
- Grid center: x=29.39, y=1.30, z=20.49
- Box size: 24 x 24 x 24 Angstrom
- Output: docked_ligand.pdbqt

In [19]:
!vina \
--receptor receptor.pdbqt \
--ligand ligand.pdbqt \
--center_x 29.39 \
--center_y 1.30 \
--center_z 20.49 \
--size_x 24 \
--size_y 24 \
--size_z 24 \
--exhaustiveness 8 \
--num_modes 3 \
--out docked_ligand.pdbqt

AutoDock Vina v1.2.3
#################################################################
# If you used AutoDock Vina in your work, please cite:          #
#                                                               #
# J. Eberhardt, D. Santos-Martins, A. F. Tillack, and S. Forli  #
# AutoDock Vina 1.2.0: New Docking Methods, Expanded Force      #
# Field, and Python Bindings, J. Chem. Inf. Model. (2021)       #
# DOI 10.1021/acs.jcim.1c00203                                  #
#                                                               #
# O. Trott, A. J. Olson,                                        #
# AutoDock Vina: improving the speed and accuracy of docking    #
# with a new scoring function, efficient optimization and       #
# multithreading, J. Comp. Chem. (2010)                         #
# DOI 10.1002/jcc.21334                                         #
#                                                               #
# Please see https://github.com/ccsb-scripps/AutoDock-V

## Step 14: Docking Results and Visualization

Binding affinity scores for all predicted poses are shown below.
The docked complex (receptor + ligand) can be visualized interactively
by loading `docked_ligand.pdbqt` into PyMOL.



In [18]:
import subprocess

# Parse docking scores from output file
result = subprocess.run(
    ["grep", "REMARK VINA RESULT", "docked_ligand.pdbqt"],
    capture_output=True, text=True
)
lines = [l for l in result.stdout.strip().split("\n") if l.strip()]

# Display results table
print("=" * 55)
print("AUTODOCK VINA — DOCKING RESULTS")
print("Receptor : TLR4-MD2 complex (PDB: 3FXI)")
print("Binding site : MD-2 hydrophobic pocket (Chain C)")
print("=" * 55)
print(f"{'Pose':<6} {'Affinity (kcal/mol)':<22} {'RMSD l.b.':<12} {'RMSD u.b.'}")
print("-" * 55)
for i, line in enumerate(lines, 1):
    parts = line.split()
    if len(parts) >= 6:
        print(f"{i:<6} {parts[3]:<22} {parts[4]:<12} {parts[5]}")
print("=" * 55)
if lines:
    best = lines[0].split()
    print(f"Best binding affinity : {best[3]} kcal/mol (Pose 1)")
print("Visualization : Load docked_ligand.pdbqt in PyMOL")
print("Static figure : see figures/TLR4_best_hit.png")
print("=" * 55)

AUTODOCK VINA — DOCKING RESULTS
Receptor : TLR4-MD2 complex (PDB: 3FXI)
Binding site : MD-2 hydrophobic pocket (Chain C)
Pose   Affinity (kcal/mol)    RMSD l.b.    RMSD u.b.
-------------------------------------------------------
Visualization : Load docked_ligand.pdbqt in PyMOL
Static figure : see figures/TLR4_best_hit.png
